In [ ]:
import pandas as pd
import numpy as np
import ast

import os
import matplotlib.pyplot as plt
import seaborn as sns
from plinder_analysis_utils import DockingAnalysisBase, PoseBustersAnalysis, PropertyAnalysis

import statsmodels.formula.api as smf


In [ ]:
PLINDER_TEST_COLUMNS = [
    "system_id", "ligand_smiles",
    # binary 
    # "ligand_is_covalent", "ligand_is_ion", "ligand_is_cofactor", "ligand_is_artifact",
    # discrete
    "system_num_protein_chains",
    "ligand_num_rot_bonds", "ligand_num_hbd", "ligand_num_hba", "ligand_num_rings",
    # continuous
    "entry_resolution", "entry_validation_molprobity", 
    "system_num_pocket_residues", "system_num_interactions",
    "ligand_molecular_weight", "ligand_crippen_clogp", 
    "ligand_num_interacting_residues", "ligand_num_neighboring_residues", "ligand_num_interactions",
]
# Create category mapping for visualization
CATEGORY_MAPPING = {
    "ligand_is_covalent": "binary",
    "ligand_is_ion": "binary",
    "ligand_is_cofactor": "binary",
    "ligand_is_artifact": "binary",
    "system_num_protein_chains": "discrete",
    "ligand_num_rot_bonds": "continuous",    
    "ligand_num_hbd": "continuous",
    "ligand_num_hba": "continuous",
    "ligand_num_rings": "continuous",
    "entry_resolution": "continuous",
    "entry_validation_molprobity": "continuous",
    "system_num_pocket_residues": "continuous",
    "system_num_interactions": "continuous",
    "ligand_molecular_weight": "continuous",
    "ligand_crippen_clogp": "continuous",
    "ligand_num_interacting_residues": "continuous",
    "ligand_num_neighboring_residues": "continuous",
    "ligand_num_interactions": "continuous",
    "ligand_is_artifact": "binary"     
}

In [ ]:
df_combined = pd.read_csv("plinder_set_0_annotated.csv")

In [ ]:
# build a boolean mask: drop any row where covalent, ionic or has_ion is True
mask = ~(
    df_combined['ligand_is_covalent'] |
    df_combined['ligand_is_ion'] |
    df_combined['has_ion'] |
    df_combined['ligand_is_cofactor']
)

# filter and reset index
df_combined = df_combined.loc[mask].reset_index(drop=True)
print("Filtered shape:", df_combined.shape)

In [ ]:
# First analyze multiple properties
property_analysis = PropertyAnalysis(df_combined)
methods = ["surfdock", "gnina", "chai-1", "diffdock_pocket_only", "icm", "vina"]

# Comparative Between Physics-based and ML-based: Mixed Effect Analysi

## Prepare the df

In [ ]:
MIXED_EFFECT_VARS = [
    "protein", "rmsd","method",
    # "system_id", "ligand_smiles",
    # binary 
    # "ligand_is_covalent", "ligand_is_ion", "ligand_is_cofactor", "ligand_is_artifact",
    # discrete
    # "system_num_protein_chains",
    "ligand_num_rot_bonds", "ligand_num_hbd", "ligand_num_hba", "ligand_num_rings",
    # continuous
    "entry_resolution", "entry_validation_molprobity", 
    # "system_num_pocket_residues", 
    "system_num_interactions",
    "ligand_molecular_weight", "ligand_crippen_clogp", 
    "ligand_num_interacting_residues", 
    "ligand_num_neighboring_residues", 
    "ligand_num_interactions",
]

In [ ]:
df_mixed = df_combined[MIXED_EFFECT_VARS]
# Create a Method_Type column based on the classification
df_mixed['Method_Type'] = df_mixed['method'].apply(
    lambda x: 'ML' if x in ['chai-1', 'diffdock_pocket_only', 'surfdock'] else 'Physics'
)

# Display the counts of each method type
print(df_mixed['Method_Type'].value_counts())

# Verify the classification
for method in df_mixed['method'].unique():
    method_type = 'ML' if method in ['chai-1', 'diffdock_pocket_only', 'surfdock'] else 'Physics'
    print(f"{method}: {method_type}")

## Mixed_Effect Analysis

### system_num_protein_chains

In [ ]:
property = "ligand_num_rot_bonds"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

model = smf.mixedlm(
    f"rmsd ~ {property} * Method_Type",
    data=df_method,
    groups=df_method["protein"]
).fit()

print(model.summary())

### ligand_num_rot_bonds

In [ ]:
property = "ligand_num_rot_bonds"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

property_analysis.analyze_comparative_fixed_effects(df_method, property, "Method_Type", "rmsd", "protein")

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "gnina", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())


#### without gnina

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

### ligand_num_hbd

In [ ]:
property = "ligand_num_hbd"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

property_analysis.analyze_comparative_fixed_effects(df_method, property, "Method_Type", "rmsd", "protein")

results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "gnina", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())


#### without gnina

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

### ligand_num_hba

In [ ]:
property = "ligand_num_hba"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

property_analysis.analyze_comparative_fixed_effects(df_method, property, "Method_Type", "rmsd", "protein")

results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "gnina", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())


#### without gnina

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

### ligand_num_rings

In [ ]:
property = "ligand_num_rings"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

property_analysis.analyze_comparative_fixed_effects(df_method, property, "Method_Type", "rmsd", "protein")

results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "gnina", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())


#### without gnina

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

### entry_resolution

In [ ]:
property = "entry_resolution"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

property_analysis.analyze_comparative_fixed_effects(df_method, property, "Method_Type", "rmsd", "protein")

results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "gnina", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())


#### without gnina

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

### entry_validation_molprobity

In [ ]:
property = "entry_validation_molprobity"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

property_analysis.analyze_comparative_fixed_effects(df_method, property, "Method_Type", "rmsd", "protein")

results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "gnina", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())


#### without gnina

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

### system_pocket_residues

### system_num_interactions

### ligand_molecular_weight

In [ ]:
property = "ligand_molecular_weight"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

property_analysis.analyze_comparative_fixed_effects(df_method, property, "Method_Type", "rmsd", "protein")

results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "gnina", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

#### without gnina

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

### ligand_crippen_clogp

In [ ]:
property = "ligand_crippen_clogp"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

property_analysis.analyze_comparative_fixed_effects(df_method, property, "Method_Type", "rmsd", "protein")

results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "gnina", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(f"p_interaction: {results['p_interaction']}")
print(results['model_full'].summary())
print(results['model_noint'].summary())


#### without gnina

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

### ligand_num_interactions

In [ ]:
property = "ligand_num_interactions"
df_method = (
    df_mixed[[
        "rmsd",
        property,
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
print(df_method.shape)

property_analysis.analyze_comparative_fixed_effects(df_method, property, "Method_Type", "rmsd", "protein")

results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "gnina", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(f"p_interaction: {results['p_interaction']}")
print(results['model_full'].summary())
print(results['model_noint'].summary())


#### without gnina

In [ ]:
results = property_analysis.analyze_continous_property_mixed_effects(
    df_combined[df_combined["method"].isin(["chai-1", "diffdock_pocket_only", "surfdock"])],
    df_combined[df_combined["method"].isin(["icm", "vina"])],
    property_name=property, 
    group1_label="ML",
    group2_label="Physics",
)
print(results['p_interaction'])
print(results['model_full'].summary())
print(results['model_noint'].summary())

### ligand_num_neighboring_residues

### ligand_num_interacting_residues

In [ ]:
df_mixed = (
    df_mixed[[
        "rmsd",
        "ligand_num_interacting_residues",
        "Method_Type",
        "protein"       # or "system_id", whichever you’re grouping by
    ]]
    .dropna()
    .reset_index(drop=True)    # <<-- important!
)
df_mixed.shape

## GAM

### A Test Case for Mixed Effect Analysis, GAM, and Batch Testing

#### 1. Mixed-Effects Linear Model (per-property)

In [ ]:
import statsmodels.formula.api as smf

model = smf.mixedlm(
    "rmsd ~ ligand_num_interacting_residues * Method_Type",
    data=df_mixed,
    groups=df_mixed["protein"]
).fit()

print(model.summary())

#### 2. Generalized Additive Model (to capture non-linearity)

In [ ]:
import numpy as np
from pygam import LinearGAM, s, f

# Encode Method_Type as 0 (Physics) / 1 (ML)
df = df_mixed.copy()
df["Method_Num"] = (df["Method_Type"] == "ML").astype(int)

X = df[["ligand_num_hba", "Method_Num"]].values
y = df["rmsd"].values

# s(0) is a spline on ligand_num_hba
# f(1) is a factor for Method_Num
# s(0, by=1) is an interaction spline (separate smooths per Method_Num)
gam = LinearGAM(s(0) + f(1) + s(0, by=1)).fit(X, y)

gam.summary()

#### 3. Batch Testing + FDR Correction (multiple properties)

In [ ]:
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

properties = [
    'ligand_num_hba',
    'ligand_num_hbd',
    'ligand_molecular_weight',
    # … add more
]

results = []
for prop in properties:
    formula = f"rmsd ~ {prop} * Method_Type"
    m = smf.mixedlm(formula, data=df_mixed, groups=df_mixed["protein"]).fit()
    p_int = m.pvalues.get(f"{prop}:Method_Type[T.ML]", np.nan)
    results.append((prop, p_int))

# Pull out p-values and correct
props, pvals = zip(*results)
rejected, pvals_corrected, _, _ = multipletests(pvals, method="fdr_bh")

for prop, p_un, p_cor, rej in zip(props, pvals, pvals_corrected, rejected):
    print(f"{prop:20s}  p_int={p_un:.3g}  p_fdr={p_cor:.3g}  significant={rej}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.lmplot(
    data=df_mixed,
    x="ligand_num_hba",
    y="rmsd",
    hue="Method_Type",
    scatter_kws={'alpha':0.3},
    height=5, aspect=1.2,
    markers=["o","s"]
)
plt.axhline(2.0, color='grey', linestyle='--', label="RMSD=2Å")
plt.title("RMSD vs. HBA, ML vs. Physics")
plt.tight_layout()
plt.show()

In [ ]:
# e.g. low (≤5), medium (6–10), high (>10) HBA
df_mixed['HBA_bin'] = pd.cut(df_mixed['ligand_num_hba'],
                                bins=[0,5,10,df_mixed.ligand_num_hba.max()],
                                labels=['low','med','high'])
summary = (
    df_mixed
    .assign(success=lambda d: d.rmsd < 2.0)
    .groupby(['HBA_bin','Method_Type'])
    .success
    .agg(['mean','count'])
    .rename(columns={'mean':'success_rate','count':'n'})
    .reset_index()
)
print(summary)